## 📦 Installation

In [ ]:
!pip install -q timm albumentations kornia
!pip install -q transformers

In [ ]:
# Clone PlantDoc dataset (only if not already present)
import os

if not os.path.isdir("PlantDoc-Dataset"):
    !git clone https://github.com/pratikkayal/PlantDoc-Dataset.git
else:
    print("PlantDoc-Dataset already exists - skipping clone")

!ls -lah PlantDoc-Dataset | head -50


Cloning into 'PlantDoc-Dataset'...
remote: Enumerating objects: 2670, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 2670 (delta 22), reused 22 (delta 22), pack-reused 2635 (from 1)
Receiving objects: 100% (2670/2670), 932.92 MiB | 42.89 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (2581/2581), done.
total 728K
drwxr-xr-x  5 root root 4.0K Feb 13 15:50 .
drwxr-xr-x  3 root root 4.0K Feb 13 15:50 ..
drwxr-xr-x  8 root root 4.0K Feb 13 15:50 .git
-rw-r--r--  1 root root  19K Feb 13 15:50 LICENSE.txt
-rw-r--r--  1 root root 684K Feb 13 15:50 PlantDoc_Examples.png
-rw-r--r--  1 root root 2.7K Feb 13 15:50 README.md
drwxr-xr-x 29 root root 4.0K Feb 13 15:50 test
drwxr-xr-x 30 root root 4.0K Feb 13 15:50 train


## 📚 Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import math
from pathlib import Path
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from collections import Counter

# Set seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


## ⚙️ Configuration

In [ ]:
class Config:
    # Paths (Kaggle)
    PV_TRAIN_DIR = '/kaggle/input/plantvillage/PlantVillage/train'
    PV_VAL_DIR   = '/kaggle/input/plantvillage/PlantVillage/val'
    PD_TRAIN_DIR = '/kaggle/working/PlantDoc-Dataset/train'
    PD_TEST_DIR  = '/kaggle/working/PlantDoc-Dataset/test'

    # Model
    MODEL_NAME = 'vit_base_patch16_224.augreg_in21k_ft_in1k'
    IMG_SIZE = 224
    UNFREEZE_LAST_N_BLOCKS = 4     # partial fine-tuning (stronger than LoRA-only)
    HEAD_DROPOUT = 0.2

    # Training
    BATCH_SIZE = 32
    EPOCHS = 10                    # early stopping; 10 epochs is usually sufficient

    LR = 1e-4
    WEIGHT_DECAY = 0.05
    WARMUP_EPOCHS = 1


    # UDA / SSL (FixMatch-style)
    MODE = 'UDA'
    UDA_WARMUP_EPOCHS = 1          # 1 epoch supervised warmup, then adapt
    RAMPUP_EPOCHS = 5              # ramp CORAL/SSL weights smoothly

    LAMBDA_CORAL_MAX = 0.25
    LAMBDA_SSL_MAX = 0.5
    PSEUDO_THRESH_START = 0.95
    PSEUDO_THRESH_END = 0.85

    PSEUDO_TEMPERATURE = 1.0       # set <1.0 to sharpen; 1.0 is safest

    # EMA teacher (stabilizes pseudo-labels)
    USE_EMA_TEACHER = True
    EMA_DECAY = 0.999

    # Supervised loss
    LABEL_SMOOTHING = 0.1

    # Augmentation
    USE_MIXUP = False              # often hurts UDA; keep OFF by default
    MIXUP_ALPHA = 0.3

    # Evaluation
    SAVE_TEST_PREDICTIONS = True
    SAVE_VAL_PREDICTIONS = False
    USE_TTA = True
    TTA_FLIP = True                # deterministic TTA: identity + hflip

    # Early stopping (uses PlantDoc-val split)
    EARLY_STOP_PATIENCE = 4


    # Other
    NUM_WORKERS = 2
    SEED = 42

config = Config()


## 🏷️ Label Mapping (PlantVillage ↔ PlantDoc)

In [ ]:
# -----------------------------
# Common labels & PlantDoc → PlantVillage mapping (WORKING)
# -----------------------------
# PlantVillage label format: "Crop___Disease"
common_labels = sorted([
    "Grape___healthy",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Blueberry___healthy",
    "Tomato___Septoria_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Cherry_(including_sour)___healthy",
    "Apple___Cedar_apple_rust",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___Late_blight",
    "Apple___Apple_scab",
    "Tomato___Early_blight",
    "Pepper,_bell___Bacterial_spot",
    "Peach___healthy",
    "Tomato___Bacterial_spot",
    "Apple___healthy",
    "Grape___Black_rot",
    "Potato___Early_blight",
    "Strawberry___healthy",
    "Tomato___healthy",
    "Pepper,_bell___healthy",
    "Potato___Late_blight",
    "Soybean___healthy",
    "Raspberry___healthy",
    "Squash___Powdery_mildew",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Leaf_Mold",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
])

# PlantDoc folder names -> PlantVillage normalized names
pd_to_pv_mapping = {
    "grape leaf": "Grape___healthy",
    "Corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
    "Blueberry leaf": "Blueberry___healthy",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Corn rust leaf": "Corn_(maize)___Common_rust_",
    "Cherry leaf": "Cherry_(including_sour)___healthy",
    "Apple rust leaf": "Apple___Cedar_apple_rust",
    "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Apple Scab Leaf": "Apple___Apple_scab",
    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot",
    "Peach leaf": "Peach___healthy",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Apple leaf": "Apple___healthy",
    "grape leaf black rot": "Grape___Black_rot",
    "Potato leaf early blight": "Potato___Early_blight",
    "Strawberry leaf": "Strawberry___healthy",
    "Tomato leaf": "Tomato___healthy",
    "Bell_pepper leaf": "Pepper,_bell___healthy",
    "Potato leaf late blight": "Potato___Late_blight",
    "Soyabean leaf": "Soybean___healthy",
    "Raspberry leaf": "Raspberry___healthy",
    "Squash Powdery mildew leaf": "Squash___Powdery_mildew",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato mold leaf": "Tomato___Leaf_Mold",
    "Corn Gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
}

def split_label(full_label):
    """Split 'Crop___Disease' into (crop, disease)."""
    parts = full_label.split('___')
    if len(parts) == 2:
        return parts[0], parts[1]
    return full_label, "unknown"

# Disease-only label space (classification targets)
disease_names = sorted({split_label(lbl)[1] for lbl in common_labels})
disease_to_idx = {d: i for i, d in enumerate(disease_names)}
idx_to_disease = {i: d for d, i in disease_to_idx.items()}
num_classes = len(disease_names)

print(f"✓ Common labels: {len(common_labels)}")
print(f"✓ PlantDoc mapping: {len(pd_to_pv_mapping)}")
print(f"✓ Disease classes: {num_classes}")


✓ Common labels: 27
✓ PlantDoc mapping: 27
✓ Disease classes: 15


## 🎨 Augmentation Pipelines (FIXED)

In [ ]:
# Albumentations v2 compatibility helpers
# Albumentations 2.x renamed/removed several parameters, e.g.:
# - RandomResizedCrop(height, width) -> size
# - GaussNoise(var_limit/mean) -> std_range/mean_range
# - CoarseDropout(max_holes/max_height/max_width/fill_value/...) -> num_holes_range/hole_*_range/fill/...
try:
    from packaging import version as _version
    _ALBU_VER = _version.parse(getattr(A, "__version__", "0"))
    _IS_ALBU2 = _ALBU_VER >= _version.parse("2.0.0")
except Exception:
    _IS_ALBU2 = False


def _random_resized_crop(img_size, scale=(0.6, 1.0), p=1.0):
    if _IS_ALBU2:
        return A.RandomResizedCrop(size=(img_size, img_size), scale=scale, p=p)
    return A.RandomResizedCrop(height=img_size, width=img_size, scale=scale, p=p)


def _gauss_noise(p=0.2):
    if _IS_ALBU2:
        # Approximate old var_limit=(10,50) (variance in pixel units) using std_range in [0,1] units.
        # sqrt(10)/255 ≈ 0.0124, sqrt(50)/255 ≈ 0.0277
        return A.GaussNoise(std_range=(0.012, 0.028), mean_range=(0.0, 0.0), p=p)
    return A.GaussNoise(var_limit=(10.0, 50.0), p=p)


def _coarse_dropout(p=0.3):
    if _IS_ALBU2:
        return A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(8, 32),
            hole_width_range=(8, 32),
            fill=0,
            p=p,
        )
    return A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0, p=p)


class StrongAugmentation:
    """Strong augmentation for robustness"""
    def __init__(self, img_size=224):
        self.transform = A.Compose([
            _random_resized_crop(img_size, scale=(0.6, 1.0), p=1.0),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.Rotate(limit=45, p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.5),

            # Color augmentation
            A.OneOf([
                A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=1.0),
                A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1.0),
                A.RGBShift(r_shift_limit=20, g_shift_limit=20, b_shift_limit=20, p=1.0),
            ], p=0.8),

            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),

            # Blur and noise
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, 7), p=1.0),
                A.MedianBlur(blur_limit=5, p=1.0),
                A.MotionBlur(blur_limit=7, p=1.0),
            ], p=0.3),

            _gauss_noise(p=0.2),

            # Occlusion
            _coarse_dropout(p=0.3),

            # Normalize
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])

    def __call__(self, image):
        if isinstance(image, Image.Image):
            image = np.array(image)
        return self.transform(image=image)['image']


class WeakAugmentation:
    """Weak augmentation for consistency"""
    def __init__(self, img_size=224):
        self.transform = A.Compose([
            A.Resize(height=img_size, width=img_size),
            A.HorizontalFlip(p=0.5),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])

    def __call__(self, image):
        if isinstance(image, Image.Image):
            image = np.array(image)
        return self.transform(image=image)['image']


class TestAugmentation:
    """Test-time augmentation"""
    def __init__(self, img_size=224):
        self.transform = A.Compose([
            A.Resize(height=img_size, width=img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])

    def __call__(self, image):
        if isinstance(image, Image.Image):
            image = np.array(image)
        return self.transform(image=image)['image']


print("✓ Augmentation pipelines created")


✓ Augmentation pipelines created


## 📊 Dataset Classes

In [ ]:
# -----------------------------
# Dataset classes (fixed)
# -----------------------------
from pathlib import Path

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def _norm_name(s: str) -> str:
    """Normalize folder names for robust matching (case/underscore/extra spaces)."""
    s = s.strip().lower().replace("_", " ")
    s = " ".join(s.split())
    return s


class PlantVillageDataset(Dataset):
    """PlantVillage dataset (source domain, labeled). Returns (strong, weak, label)."""
    def __init__(self, root_dir, transform_strong, transform_weak, common_labels, disease_to_idx):
        self.root_dir = Path(root_dir)
        self.transform_strong = transform_strong
        self.transform_weak = transform_weak
        self.disease_to_idx = disease_to_idx

        self.samples = []
        missing = []

        for cls in sorted(os.listdir(self.root_dir)):
            cls_dir = self.root_dir / cls
            if not cls_dir.is_dir():
                continue
            # PlantVillage folders should already be in PlantVillage label format
            if cls not in common_labels:
                continue

            _, disease = split_label(cls)
            if disease not in disease_to_idx:
                missing.append((cls, disease))
                continue
            y = disease_to_idx[disease]

            for img_path in cls_dir.iterdir():
                if img_path.is_file() and img_path.suffix.lower() in IMG_EXTS:
                    self.samples.append((str(img_path), y))

        print(f"PlantVillage: Loaded {len(self.samples)} images")
        if len(self.samples) == 0:
            print(f"WARNING: PlantVillage: found 0 images under {root_dir}")
            print("Tip: verify config.PV_*_DIR paths and folder structure.")
        if missing:
            print("WARNING: Some PlantVillage classes had diseases not in disease_to_idx (first 10):", missing[:10])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        img_strong = self.transform_strong(image)
        img_weak = self.transform_weak(image)
        return img_strong, img_weak, label


class PlantDocDataset(Dataset):
    """PlantDoc dataset (target domain). If labeled=False -> label=-1.
    If labeled=True -> uses pd_to_pv_mapping to map PlantDoc folder -> PlantVillage label -> disease index.
    Returns (strong, weak, label).
    """
    def __init__(self, root_dir, transform_strong, transform_weak, pd_to_pv_mapping, disease_to_idx, labeled=False):
        self.root_dir = Path(root_dir)
        self.transform_strong = transform_strong
        self.transform_weak = transform_weak
        self.disease_to_idx = disease_to_idx
        self.labeled = labeled

        # Build normalized mapping for robust folder-name matching
        norm_map = {_norm_name(k): v for k, v in pd_to_pv_mapping.items()}

        self.samples = []
        unmatched_folders = []
        unmapped_diseases = []

        for pd_cls in sorted(os.listdir(self.root_dir)):
            cls_dir = self.root_dir / pd_cls
            if not cls_dir.is_dir():
                continue

            y = -1
            if labeled:
                pv_label = pd_to_pv_mapping.get(pd_cls)
                if pv_label is None:
                    pv_label = norm_map.get(_norm_name(pd_cls))

                if pv_label is None:
                    unmatched_folders.append(pd_cls)
                    continue

                _, disease = split_label(pv_label)
                y = disease_to_idx.get(disease, -1)
                if y == -1:
                    unmapped_diseases.append((pd_cls, pv_label, disease))
                    continue

            for img_path in cls_dir.iterdir():
                if img_path.is_file() and img_path.suffix.lower() in IMG_EXTS:
                    self.samples.append((str(img_path), y))

        print(f"PlantDoc: Loaded {len(self.samples)} images (labeled={labeled})")
        if len(self.samples) == 0:
            print(f"WARNING: PlantDoc: found 0 images under {root_dir}")
            if unmatched_folders:
                print("Unmatched PlantDoc folders (first 20):", unmatched_folders[:20])
            if unmapped_diseases:
                print("Folders mapped but disease not in disease_to_idx (first 10):", unmapped_diseases[:10])
            print("Tip: check that PlantDoc folders match pd_to_pv_mapping keys (case/underscores/spaces) and that mapped PV labels exist in common_labels.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        img_strong = self.transform_strong(image)
        img_weak = self.transform_weak(image)
        return img_strong, img_weak, label


## 🧠 Model with LoRA

In [ ]:
import timm
from copy import deepcopy

class ViTClassifier(nn.Module):
    """ViT classifier with partial fine-tuning (unfreeze last N blocks)."""
    def __init__(self, model_name: str, num_classes: int, unfreeze_last_n: int = 4, head_dropout: float = 0.2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.feature_dim = self.backbone.num_features

        # Freeze everything first
        for p in self.backbone.parameters():
            p.requires_grad = False

        # Unfreeze last transformer blocks + final norm (common best practice for ViT transfer)
        if hasattr(self.backbone, "blocks"):
            for blk in self.backbone.blocks[-unfreeze_last_n:]:
                for p in blk.parameters():
                    p.requires_grad = True
        if hasattr(self.backbone, "norm"):
            for p in self.backbone.norm.parameters():
                p.requires_grad = True

        # Classification head
        self.head = nn.Sequential(
            nn.LayerNorm(self.feature_dim),
            nn.Dropout(head_dropout),
            nn.Linear(self.feature_dim, num_classes)
        )

    def forward(self, x, return_features=False):
        feats = self.backbone(x)  # pooled features
        logits = self.head(feats)
        if return_features:
            return logits, feats
        return logits


@torch.no_grad()
def ema_update(teacher: nn.Module, student: nn.Module, decay: float):
    """EMA teacher update: theta_t <- decay*theta_t + (1-decay)*theta_s"""
    for t_p, s_p in zip(teacher.parameters(), student.parameters()):
        t_p.data.mul_(decay).add_(s_p.data, alpha=1.0 - decay)


## 🎯 Loss Functions

In [ ]:
def coral_loss(source_features, target_features):
    """CORAL: Correlation Alignment"""
    # Center features
    source = source_features - source_features.mean(0, keepdim=True)
    target = target_features - target_features.mean(0, keepdim=True)

    n_s, d = source.shape
    n_t = target.shape[0]

    # Covariance matrices
    cov_s = (source.t() @ source) / (n_s - 1)
    cov_t = (target.t() @ target) / (n_t - 1)

    # Frobenius norm
    loss = torch.norm(cov_s - cov_t, p='fro') ** 2
    loss = loss / (4 * d * d)

    return loss


def contrastive_loss(feat1, feat2, temperature=0.5):
    """NT-Xent contrastive loss (SimCLR)"""
    batch_size = feat1.size(0)

    # Normalize
    feat1 = F.normalize(feat1, dim=1)
    feat2 = F.normalize(feat2, dim=1)

    # Concatenate
    features = torch.cat([feat1, feat2], dim=0)

    # Similarity matrix
    similarity = (features @ features.t()) / temperature

    # Mask self-similarity
    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=features.device)
    similarity = similarity.masked_fill(mask, -1e9)

    # Labels: positive pairs
    labels = torch.cat([
        torch.arange(batch_size, 2*batch_size),
        torch.arange(batch_size)
    ], dim=0).to(features.device)

    loss = F.cross_entropy(similarity, labels)
    return loss


def mixup_data(x, y, alpha=0.2):
    """MixUp augmentation"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam


class LabelSmoothingCE(nn.Module):
    def __init__(self, epsilon=0.1):
        super().__init__()
        self.epsilon = epsilon

    def forward(self, logits, target):
        n_classes = logits.size(-1)
        log_probs = F.log_softmax(logits, dim=-1)

        # One-hot with smoothing
        targets = torch.zeros_like(log_probs)
        targets.scatter_(1, target.unsqueeze(1), 1)
        targets = targets * (1 - self.epsilon) + self.epsilon / n_classes

        loss = -(targets * log_probs).sum(dim=-1).mean()
        return loss

print("✓ Loss functions defined")

✓ Loss functions defined


## 📂 Load Data

In [ ]:
# Create augmentations
aug_strong = StrongAugmentation(config.IMG_SIZE)
aug_weak = WeakAugmentation(config.IMG_SIZE)
aug_test = TestAugmentation(config.IMG_SIZE)

# Create datasets
print("Loading datasets...\n")

# Source (PlantVillage)
source_train = PlantVillageDataset(
    config.PV_TRAIN_DIR,
    aug_strong, aug_weak,
    common_labels,
    disease_to_idx
)

# Target (PlantDoc) - unlabeled for UDA
if config.MODE == 'UDA':
    target_train = PlantDocDataset(
        config.PD_TRAIN_DIR,
        aug_strong, aug_weak,
        pd_to_pv_mapping,
        disease_to_idx,
        labeled=False
    )

    if len(target_train) == 0:
        raise ValueError(
            f"PlantDoc target_train is empty (0 images). Check config.PD_TRAIN_DIR={config.PD_TRAIN_DIR} and pd_to_pv_mapping keys vs folder names."
        )

# Test (PlantDoc) - labeled
test_dataset = PlantDocDataset(
    config.PD_TEST_DIR,
    aug_test, aug_test,
    pd_to_pv_mapping,
    disease_to_idx,
    labeled=True
)

# DataLoaders
source_loader = DataLoader(
    source_train,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

if config.MODE == 'UDA':
    target_loader = DataLoader(
        target_train,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=config.NUM_WORKERS,
        pin_memory=True
    )
else:
    target_loader = None

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

print(f"\n✓ Data loaded successfully")
print(f"  Source train: {len(source_train)}")
if config.MODE == 'UDA':
    print(f"  Target train: {len(target_train)}")
print(f"  Test: {len(test_dataset)}")

Loading datasets...

PlantVillage: Loaded 29494 images
PlantDoc: Loaded 2342 images (labeled=False)
PlantDoc: Loaded 236 images (labeled=True)

✓ Data loaded successfully
  Source train: 29494
  Target train: 2342
  Test: 236


In [ ]:
def linear_rampup(epoch, warmup_epochs, rampup_epochs):
    """0 during warmup, then linearly increases to 1."""
    if epoch <= warmup_epochs:
        return 0.0
    t = (epoch - warmup_epochs) / float(max(1, rampup_epochs))
    return float(min(1.0, max(0.0, t)))

def pseudo_threshold(epoch, start, end, total_epochs):
    """Linearly anneal threshold from start -> end."""
    if total_epochs <= 1:
        return end
    t = (epoch - 1) / float(total_epochs - 1)
    return float(start + t * (end - start))

@torch.no_grad()
def update_da_prob(da_prob, probs, momentum=0.99):
    """Running average of target class probabilities for distribution alignment."""
    batch_mean = probs.mean(dim=0)
    da_prob.mul_(momentum).add_(batch_mean, alpha=1.0 - momentum)
    da_prob.clamp_(min=1e-6)
    da_prob.div_(da_prob.sum())
    return da_prob

def distribution_alignment(probs, da_prob, source_prior):
    """Adjust probs so target marginal matches source prior (simple DA)."""
    adj = probs / da_prob.unsqueeze(0)
    adj = adj * source_prior.unsqueeze(0)
    adj = adj / adj.sum(dim=1, keepdim=True)
    return adj

def train_epoch_fixmatch(student, teacher, source_loader, target_loader, optimizer, scheduler, scaler, epoch, config, da_state):
    student.train()
    if teacher is not None:
        teacher.eval()

    criterion_ce = nn.CrossEntropyLoss(label_smoothing=config.LABEL_SMOOTHING)

    total_loss = 0.0
    ce_sum = 0.0
    coral_sum = 0.0
    ssl_sum = 0.0
    mask_ratio_sum = 0.0

    correct = 0
    total = 0

    target_iter = iter(target_loader) if target_loader is not None else None
    pbar = tqdm(source_loader, desc=f"Epoch {epoch}")

    # Ramp weights + threshold schedule
    ramp = linear_rampup(epoch, config.UDA_WARMUP_EPOCHS, config.RAMPUP_EPOCHS)
    w_coral = ramp * config.LAMBDA_CORAL_MAX
    w_ssl = ramp * config.LAMBDA_SSL_MAX
    thr = pseudo_threshold(epoch, config.PSEUDO_THRESH_START, config.PSEUDO_THRESH_END, config.EPOCHS)

    for x_s_strong, x_s_weak, y_s in pbar:
        x_s_strong = x_s_strong.to(device, non_blocking=True)
        x_s_weak   = x_s_weak.to(device, non_blocking=True)
        y_s        = y_s.to(device, non_blocking=True)

        # Get a target batch
        if target_iter is not None:
            try:
                x_t_strong, x_t_weak, _ = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader)
                x_t_strong, x_t_weak, _ = next(target_iter)

            x_t_strong = x_t_strong.to(device, non_blocking=True)
            x_t_weak   = x_t_weak.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=(device == "cuda")):
            # Source supervised
            logits_s, feat_s = student(x_s_strong, return_features=True)
            loss_ce = criterion_ce(logits_s, y_s)

            # Source accuracy (only when not using mixup)
            pred = logits_s.argmax(dim=1)
            correct += (pred == y_s).sum().item()
            total += y_s.size(0)

            loss_coral = torch.tensor(0.0, device=device)
            loss_ssl = torch.tensor(0.0, device=device)
            mask_ratio = 0.0

            if target_iter is not None and ramp > 0.0:
                # -----------------------------
                # CORAL feature alignment
                # -----------------------------
                if w_coral > 0.0:
                    # Features for CORAL (use weak views for stability)
                    _, feat_s_weak = student(x_s_weak, return_features=True)
                    _, feat_t_weak = student(x_t_weak, return_features=True)
                    loss_coral = coral_loss(feat_s_weak, feat_t_weak)

                # -----------------------------
                # FixMatch-style pseudo-labeling (SSL)
                # -----------------------------
                if w_ssl > 0.0:
                    with torch.no_grad():
                        if teacher is None:
                            t_logits = student(x_t_weak)
                        else:
                            t_logits = teacher(x_t_weak)

                        t_probs = F.softmax(t_logits / config.PSEUDO_TEMPERATURE, dim=1)

                        # Distribution alignment (optional but helpful)
                        if da_state is not None:
                            da_prob = da_state['da_prob']
                            source_prior = da_state['source_prior']
                            da_prob = update_da_prob(da_prob, t_probs, momentum=0.99)
                            t_probs = distribution_alignment(t_probs, da_prob, source_prior)
                            da_state['da_prob'] = da_prob

                        max_probs, pseudo = torch.max(t_probs, dim=1)
                        mask = max_probs.ge(thr).float()
                        mask_ratio = float(mask.mean().item())

                    if mask_ratio > 0:
                        s_logits = student(x_t_strong)
                        loss_ssl = (F.cross_entropy(s_logits, pseudo, reduction='none') * mask).mean()

            loss = loss_ce + w_coral * loss_coral + w_ssl * loss_ssl

        # Backprop (AMP)
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        # EMA update
        if teacher is not None:
            ema_update(teacher, student, config.EMA_DECAY)

        # Logging
        total_loss += float(loss.item())
        ce_sum += float(loss_ce.item())
        coral_sum += float(loss_coral.item())
        ssl_sum += float(loss_ssl.item())
        mask_ratio_sum += mask_ratio

        pbar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'ce': f"{loss_ce.item():.4f}",
            'coral': f"{loss_coral.item():.4f}",
            'ssl': f"{loss_ssl.item():.4f}",
            'mask': f"{mask_ratio*100:.1f}%",
            'acc': f"{100*correct/total:.1f}%"
        })

    n = len(source_loader)
    avg_loss = total_loss / n
    avg_ce = ce_sum / n
    avg_coral = coral_sum / n
    avg_ssl = ssl_sum / n
    avg_mask = mask_ratio_sum / n
    acc = 100.0 * correct / max(1, total)

    return avg_loss, avg_ce, avg_coral, avg_ssl, avg_mask, acc


from torch.utils.data import Subset
import re

def _extract_paths_in_loader_order(loader):
    """Best-effort recovery of image paths in the same order the loader yields samples.
    Works for PlantDocDataset and Subset(PlantDocDataset) when shuffle=False.
    """
    ds = loader.dataset
    # Subset case
    if isinstance(ds, Subset) and hasattr(ds, "dataset") and hasattr(ds, "indices"):
        base = ds.dataset
        idxs = list(ds.indices)
        if hasattr(base, "samples"):
            return [base.samples[i][0] for i in idxs]
        return None
    # Direct dataset case
    if hasattr(ds, "samples"):
        return [p for p, _ in ds.samples]
    return None


@torch.no_grad()
def evaluate_model(model, loader, use_tta=False, tta_flip=True, return_probs=False):
    """Evaluate accuracy/F1 on a labeled loader.
    Loader must yield (x_strong, x_weak, y). If return_probs=True, also returns NxC probabilities.
    """
    model.eval()
    all_preds = []
    all_labels = []
    prob_chunks = []

    for x_strong, x_weak, labels in tqdm(loader, desc="Eval"):
        x = x_strong.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if use_tta and tta_flip:
            logits1 = model(x)
            logits2 = model(torch.flip(x, dims=[3]))
            logits = (logits1 + logits2) / 2.0
        else:
            logits = model(x)

        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

        if return_probs:
            probs = F.softmax(logits, dim=1)
            prob_chunks.append(probs.cpu())

    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    f1_weighted = f1_score(all_labels, all_preds, average='weighted')

    if return_probs:
        all_probs = torch.cat(prob_chunks, dim=0).numpy() if prob_chunks else np.zeros((0, 0), dtype=np.float32)
        return acc, f1_macro, f1_weighted, all_preds, all_labels, all_probs

    return acc, f1_macro, f1_weighted, all_preds, all_labels


def save_detailed_predictions_csv(csv_path, loader, y_true, y_pred, probs, class_names):
    """Save per-sample predictions for statistical comparisons (paired tests, McNemar, etc.)."""
    paths = _extract_paths_in_loader_order(loader)
    n = len(y_true)
    if paths is None:
        paths = [None] * n
    elif len(paths) != n:
        # Safety fallback (shouldn't happen if shuffle=False)
        paths = (paths[:n] + [None] * max(0, n - len(paths)))

    df = pd.DataFrame({
        "sample_id": np.arange(n, dtype=int),
        "image_path": paths,
        "y_true": np.array(y_true, dtype=int),
        "y_pred": np.array(y_pred, dtype=int),
    })
    df["true_name"] = df["y_true"].map(lambda i: class_names[i] if 0 <= i < len(class_names) else "unknown")
    df["pred_name"] = df["y_pred"].map(lambda i: class_names[i] if 0 <= i < len(class_names) else "unknown")

    # confidence = max probability
    if probs is not None and probs.size > 0:
        df["pred_conf"] = probs.max(axis=1)
        # add per-class probabilities (stable column order)
        # add per-class probabilities (sanitized column names)
        used = set(df.columns)
        for j, cname in enumerate(class_names):
            safe = re.sub(r'[^0-9a-zA-Z]+', '_', str(cname)).strip('_')
            col = f"p_{safe}"
            # avoid accidental duplicates
            k = 2
            while col in used:
                col = f"p_{safe}_{k}"
                k += 1
            used.add(col)
            df[col] = probs[:, j]
    else:
        df["pred_conf"] = np.nan

    df.to_csv(csv_path, index=False)
    return df


## 🧪 Ablation Study (5 epochs each)

This section runs **6 ablation configurations**, each for **exactly 5 epochs**, using the **same PlantDoc val/test split** for fair comparison.

For each run, the notebook saves:

- `training_history.csv`
- `val_predictions.csv` (optional; enabled below)
- `test_predictions.csv` (**detailed per-sample predictions + probabilities**, for statistical tests)
- `results.json` (summary metrics)

Outputs are written under: `ablation_runs/<EXP_NAME>/`


In [ ]:

# ============================================================
# Ablation runner (6 runs, 5 epochs each) + CSV export
# ============================================================
import os, json, gc, time, math
from copy import deepcopy

from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Subset
from torch.cuda.amp import autocast, GradScaler

# -----------------------------
# Global ablation settings
# -----------------------------
ABLATION_EPOCHS = 5
SAVE_VAL_PREDICTIONS = True   # keep True if you want paired tests on val too
RUN_DIR = "ablation_runs"
os.makedirs(RUN_DIR, exist_ok=True)

# Make a fixed PlantDoc val/test split ONCE (fair comparisons)
labels = [lbl for _, lbl in test_dataset.samples]  # PlantDoc labeled set
indices = np.arange(len(labels))
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.8, random_state=config.SEED)
val_idx, test_idx = next(splitter.split(indices, labels))

val_dataset = Subset(test_dataset, val_idx)
final_test_dataset = Subset(test_dataset, test_idx)

print(f"PlantDoc labeled split (fixed): val={len(val_dataset)} | test={len(final_test_dataset)}")

def make_eval_loaders(cfg):
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True
    )
    test_loader = DataLoader(
        final_test_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True
    )
    return val_loader, test_loader

def make_train_loaders(cfg, use_target: bool):
    # Fresh loaders per run (reproducibility + avoids state bleed)
    g = torch.Generator()
    g.manual_seed(cfg.SEED)

    src_loader = DataLoader(
        source_train,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        generator=g
    )

    if use_target:
        if 'target_train' not in globals() or target_train is None or len(target_train) == 0:
            raise ValueError("Target (PlantDoc train) is missing/empty, but this ablation requires it.")
        tgt_loader = DataLoader(
            target_train,
            batch_size=cfg.BATCH_SIZE,
            shuffle=True,
            num_workers=cfg.NUM_WORKERS,
            pin_memory=True,
            generator=g
        )
    else:
        tgt_loader = None

    return src_loader, tgt_loader

def build_models_and_optim(cfg, src_loader):
    # Re-seed for each run
    set_seed(cfg.SEED)

    # Student
    student = ViTClassifier(
        cfg.MODEL_NAME,
        num_classes,
        unfreeze_last_n=cfg.UNFREEZE_LAST_N_BLOCKS,
        head_dropout=cfg.HEAD_DROPOUT
    ).to(device)

    # Teacher (EMA)
    if cfg.USE_EMA_TEACHER:
        teacher = deepcopy(student).to(device)
        for p in teacher.parameters():
            p.requires_grad = False
        teacher.eval()
    else:
        teacher = None

    # Optimizer
    optimizer = torch.optim.AdamW(
        [p for p in student.parameters() if p.requires_grad],
        lr=cfg.LR,
        weight_decay=cfg.WEIGHT_DECAY
    )

    # Step-based warmup + cosine schedule
    steps_per_epoch = len(src_loader)
    num_steps = steps_per_epoch * cfg.EPOCHS
    warmup_steps = steps_per_epoch * cfg.WARMUP_EPOCHS

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, num_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # AMP scaler
    scaler = GradScaler(enabled=(device == "cuda"))

    return student, teacher, optimizer, scheduler, scaler

def init_da_state(cfg):
    # distribution alignment state (source prior from PlantVillage)
    src_labels = [lbl for _, lbl in source_train.samples]
    src_counts = np.bincount(src_labels, minlength=num_classes).astype(np.float64)
    src_prior = src_counts / max(1.0, src_counts.sum())
    source_prior = torch.tensor(src_prior, device=device, dtype=torch.float32)

    return {
        'da_prob': torch.ones(num_classes, device=device, dtype=torch.float32) / num_classes,
        'source_prior': source_prior
    }

def run_one_experiment(exp_name: str, overrides: dict, use_target: bool):
    out_dir = os.path.join(RUN_DIR, exp_name)
    os.makedirs(out_dir, exist_ok=True)

    # Build a per-run config object (do not mutate global Config)
    cfg = deepcopy(config)
    for k, v in overrides.items():
        setattr(cfg, k, v)
    cfg.EPOCHS = ABLATION_EPOCHS

    # ------------------------------------------------------------
    # IMPORTANT: training code in this notebook uses the global `config`
    # object inside helper functions (e.g., train_epoch_fixmatch).
    # To ensure ablation overrides actually take effect, we temporarily
    # bind the global `config` to this per-run cfg.
    # ------------------------------------------------------------
    _old_global_config = globals().get("config", None)
    globals()["config"] = cfg

    # Loaders
    src_loader, tgt_loader = make_train_loaders(cfg, use_target=use_target)
    val_loader, test_loader = make_eval_loaders(cfg)

    # Model + optim
    student, teacher, optimizer, scheduler, scaler = build_models_and_optim(cfg, src_loader)

    # DA state only matters if SSL is active
    da_state = init_da_state(cfg) if cfg.LAMBDA_SSL_MAX > 0 else None

    # Run exactly cfg.EPOCHS epochs; track best by val macro-F1
    best_val_f1 = -1.0
    best_epoch = -1
    history = []

    print("\n" + "="*70)
    print(f"RUN: {exp_name}")
    print(f"Epochs={cfg.EPOCHS} | L_SSL={cfg.LAMBDA_SSL_MAX} | L_CORAL={cfg.LAMBDA_CORAL_MAX} | EMA={cfg.USE_EMA_TEACHER}")
    print(f"Thr: {cfg.PSEUDO_THRESH_START} -> {cfg.PSEUDO_THRESH_END} | TTA={cfg.USE_TTA}")
    print("="*70)

    for epoch in range(1, cfg.EPOCHS + 1):
        avg_loss, avg_ce, avg_coral, avg_ssl, avg_mask, train_acc = train_epoch_fixmatch(
            student, teacher, src_loader, tgt_loader, optimizer, scheduler, scaler, epoch, cfg, da_state
        )

        eval_model = teacher if (teacher is not None) else student
        val_acc, val_f1_macro, val_f1_weighted, val_preds, val_labels, val_probs = evaluate_model(
            eval_model, val_loader, use_tta=cfg.USE_TTA, tta_flip=cfg.TTA_FLIP, return_probs=True
        )

        print(f"\nEpoch {epoch}/{cfg.EPOCHS}")
        print(f"Train: Loss={avg_loss:.4f}, CE={avg_ce:.4f}, CORAL={avg_coral:.4f}, SSL={avg_ssl:.4f}, Mask={avg_mask*100:.1f}%, Acc={train_acc:.1f}%")
        print(f"Val:   Acc={val_acc*100:.2f}%, F1-macro={val_f1_macro*100:.2f}%, F1-weighted={val_f1_weighted*100:.2f}%")

        history.append({
            "epoch": epoch,
            "train_loss": avg_loss,
            "train_ce": avg_ce,
            "train_coral": avg_coral,
            "train_ssl": avg_ssl,
            "mask_ratio": avg_mask,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "val_f1_macro": val_f1_macro,
            "val_f1_weighted": val_f1_weighted,
        })

        # Save best checkpoint (in out_dir)
        if val_f1_macro > best_val_f1:
            best_val_f1 = val_f1_macro
            best_epoch = epoch
            torch.save(student.state_dict(), os.path.join(out_dir, "best_student.pth"))
            if teacher is not None:
                torch.save(teacher.state_dict(), os.path.join(out_dir, "best_teacher.pth"))
            print("✓ Best model updated (val F1-macro).")

        # Optional: save val predictions each epoch? (Usually no; we save only best epoch later)
        # Keep loop simple for speed.

    # Save training history
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(os.path.join(out_dir, "training_history.csv"), index=False)

    # Load best checkpoint and evaluate on val + test (once)
    if teacher is not None and os.path.exists(os.path.join(out_dir, "best_teacher.pth")):
        teacher.load_state_dict(torch.load(os.path.join(out_dir, "best_teacher.pth"), map_location=device))
        final_model = teacher
    else:
        student.load_state_dict(torch.load(os.path.join(out_dir, "best_student.pth"), map_location=device))
        final_model = student

    # Final Val predictions (best epoch)
    if SAVE_VAL_PREDICTIONS:
        v_acc, v_f1m, v_f1w, v_preds, v_labels, v_probs = evaluate_model(
            final_model, val_loader, use_tta=cfg.USE_TTA, tta_flip=cfg.TTA_FLIP, return_probs=True
        )
        save_detailed_predictions_csv(
            os.path.join(out_dir, "val_predictions.csv"),
            val_loader,
            y_true=v_labels,
            y_pred=v_preds,
            probs=v_probs,
            class_names=disease_names
        )

    # Test predictions (best epoch)
    test_acc, test_f1_macro, test_f1_weighted, preds, labels, test_probs = evaluate_model(
        final_model, test_loader, use_tta=cfg.USE_TTA, tta_flip=cfg.TTA_FLIP, return_probs=True
    )

    save_detailed_predictions_csv(
        os.path.join(out_dir, "test_predictions.csv"),
        test_loader,
        y_true=labels,
        y_pred=preds,
        probs=test_probs,
        class_names=disease_names
    )

    # Summary JSON
    summary = {
        "exp_name": exp_name,
        "epochs": cfg.EPOCHS,
        "best_epoch_by_val_f1_macro": int(best_epoch),
        "best_val_f1_macro": float(best_val_f1),
        "test_acc": float(test_acc),
        "test_f1_macro": float(test_f1_macro),
        "test_f1_weighted": float(test_f1_weighted),
        "use_tta": bool(cfg.USE_TTA),
        "tta_flip": bool(cfg.TTA_FLIP),
        "lambda_ssl_max": float(cfg.LAMBDA_SSL_MAX),
        "lambda_coral_max": float(cfg.LAMBDA_CORAL_MAX),
        "use_ema_teacher": bool(cfg.USE_EMA_TEACHER),
        "pseudo_thresh_start": float(cfg.PSEUDO_THRESH_START),
        "pseudo_thresh_end": float(cfg.PSEUDO_THRESH_END),
        "seed": int(cfg.SEED),
    }
    with open(os.path.join(out_dir, "results.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\n✓ [{exp_name}] Test: Acc={test_acc*100:.2f}%, F1-macro={test_f1_macro*100:.2f}%, F1-weighted={test_f1_weighted*100:.2f}%")
    print(f"✓ Saved: {out_dir}/test_predictions.csv (detailed, with probabilities)")


    # Restore global config (avoid cross-run contamination)
    if _old_global_config is not None:
        globals()["config"] = _old_global_config

    # Cleanup (important on Kaggle)
    del student, teacher, optimizer, scheduler, scaler, final_model
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    return summary

# ============================================================
# Define 6 ablation runs (exactly as requested)
# NOTE: We treat "no EMA vs EMA" as A4 vs A5.
#       We treat "fixed thr vs scheduled thr" as A5 vs A6.
# ============================================================
EXPERIMENTS = [
    # 1) Source-only (PV trained, test on PD)
    ("A1_source_only", dict(LAMBDA_CORAL_MAX=0.0, LAMBDA_SSL_MAX=0.0, USE_EMA_TEACHER=False), False),

    # 2) CORAL only (PV supervised + CORAL, no SSL)
    ("A2_coral_only", dict(LAMBDA_CORAL_MAX=0.25, LAMBDA_SSL_MAX=0.0, USE_EMA_TEACHER=False), True),

    # 3) FixMatch only (PV supervised + SSL, no CORAL)
    ("A3_fixmatch_only", dict(LAMBDA_CORAL_MAX=0.0, LAMBDA_SSL_MAX=0.5, USE_EMA_TEACHER=False), True),

    # 4) FixMatch + CORAL (no EMA)  ---> used for "no EMA vs EMA"
    ("A4_fixmatch_coral_noEMA", dict(LAMBDA_CORAL_MAX=0.25, LAMBDA_SSL_MAX=0.5, USE_EMA_TEACHER=False), True),

    # 5) FixMatch + CORAL + EMA (scheduled threshold) ---> full combo baseline for ablations
    ("A5_fixmatch_coral_EMA_schedThr", dict(LAMBDA_CORAL_MAX=0.25, LAMBDA_SSL_MAX=0.5, USE_EMA_TEACHER=True,
                                            PSEUDO_THRESH_START=0.95, PSEUDO_THRESH_END=0.85), True),

    # 6) FixMatch + CORAL + EMA (fixed threshold) ---> used for "fixed vs scheduled"
    ("A6_fixmatch_coral_EMA_fixedThr", dict(LAMBDA_CORAL_MAX=0.25, LAMBDA_SSL_MAX=0.5, USE_EMA_TEACHER=True,
                                            PSEUDO_THRESH_START=0.90, PSEUDO_THRESH_END=0.90), True),
]

# ============================================================
# Run all experiments
# ============================================================
all_summaries = []
t0 = time.time()
for exp_name, overrides, use_target in EXPERIMENTS:
    s = run_one_experiment(exp_name, overrides, use_target=use_target)
    all_summaries.append(s)

summary_df = pd.DataFrame(all_summaries).sort_values("exp_name")
summary_df.to_csv(os.path.join(RUN_DIR, "ablation_summary.csv"), index=False)

print("\n" + "="*70)
print("ABLATION STUDY COMPLETE")
print(f"✓ Saved {RUN_DIR}/ablation_summary.csv")
print(f"Total wall time (this Kaggle run): {(time.time()-t0)/3600:.2f} hours")
print("="*70)

# Display summary table
summary_df


PlantDoc labeled split (fixed): val=47 | test=189


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]


RUN: A1_source_only
Epochs=5 | L_SSL=0.0 | L_CORAL=0.0 | EMA=False
Thr: 0.95 -> 0.85 | TTA=True


Epoch 1:   0%|          | 0/922 [00:00<?, ?it/s]

Eval:   0%|          | 0/2 [00:00<?, ?it/s]

...


Epoch 5/5
Train: Loss=0.5932, CE=0.5744, CORAL=0.0038, SSL=0.0450, Mask=60.8%, Acc=99.4%
Val:   Acc=57.45%, F1-macro=34.23%, F1-weighted=48.28%


Eval:   0%|          | 0/2 [00:00<?, ?it/s]

Eval:   0%|          | 0/6 [00:00<?, ?it/s]


✓ [A6_fixmatch_coral_EMA_fixedThr] Test: Acc=71.43%, F1-macro=57.25%, F1-weighted=67.68%
✓ Saved: ablation_runs/A6_fixmatch_coral_EMA_fixedThr/test_predictions.csv (detailed, with probabilities)

ABLATION STUDY COMPLETE
✓ Saved ablation_runs/ablation_summary.csv
Total wall time (this Kaggle run): 7.57 hours


,exp_name,epochs,best_epoch_by_val_f1_macro,best_val_f1_macro,test_acc,test_f1_macro,test_f1_weighted,use_tta,tta_flip,lambda_ssl_max,lambda_coral_max,use_ema_teacher,pseudo_thresh_start,pseudo_thresh_end,seed
0,A1_source_only,5,5,0.504286,0.698413,0.544000,0.680118,True,True,0.0,0.00,False,0.95,0.85,42
1,A2_coral_only,5,5,0.517087,0.682540,0.539277,0.649033,True,True,0.0,0.25,False,0.95,0.85,42
2,A3_fixmatch_only,5,1,0.432332,0.671958,0.535729,0.656988,True,True,0.5,0.00,False,0.95,0.85,42
3,A4_fixmatch_coral_noEMA,5,1,0.432332,0.671958,0.535729,0.656988,True,True,0.5,0.25,False,0.95,0.85,42
4,A5_fixmatch_coral_EMA_schedThr,5,4,0.524092,0.703704,0.543438,0.664502,True,True,0.5,0.25,True,0.95,0.85,42
5,A6_fixmatch_coral_EMA_fixedThr,5,3,0.529206,0.714286,0.572543,0.676790,True,True,0.5,0.25,True,0.90,0.90,42
